# Protein-peptide modeling tutorial using a local version of HADDOCK3


---
## Introduction

In this tutorial, we will focus on docking the N-terminal peptide of p53 to the mouse MDM2 protein. The tumor suppressor p53 is often referred to as the guardian of the genome because of its central role in preventing tumor development. As a transcription factor, it regulates the cell cycle and can induce DNA repair or apoptosis in response to cellular stress. MDM2, an E3 ubiquitin ligase, is a key negative regulator of p53: by binding to its N-terminal transactivation domain (TAD) peptide, it inhibits p53 activity and promotes its degradation. The p53–MDM2 interaction is therefore of major biological and medical importance.

For the mouse MDM2–p53 system, no experimental structure of the complex is available. Modelling such interactions is challenging because protein–peptide docking is generally more difficult than protein–protein docking. This is primarily due to the intrinsic flexibility of peptides, which allows them to adopt multiple conformations, complicating prediction of the bound state.

In this tutorial, we will use HADDOCK3 to model the MDM2–p53 complex. Docking will be guided by pre-defined restraints derived from known interfaces of the human homolog of MDM2. As input structures, we will use an AlphaFold model of mouse MDM2 together with an ensemble of idealized peptide conformations. This combination will allow us to sample and predict plausible binding modes for the p53 peptide. Since no experimentally solved structure is available this complex, the human MDM2–p53 complex will be used as a reference for assessing the docking results (PDB ID: [1YCR](https://www.rcsb.org/structure/1YCR)).


<figure style="text-align: center;">
  <img
src="https://www.bonvinlab.org/education/HADDOCK3/HADDOCK3-protein-peptide/png/1ycr_pretty.png">
</figure>


---
# **A brief introduction to HADDOCK3**

HADDOCK3 is the next generation integrative modelling software in the
long-lasting HADDOCK project. It represents a complete rethinking and rewriting
of the HADDOCK2.X series, implementing a new way to interact with HADDOCK and
offering new features to users who can now define custom workflows.

In the previous HADDOCK2.x versions, users had access to a highly
parameterisable yet rigid simulation pipeline composed of three steps:
`rigid-body docking (it0)`, `semi-flexible refinement (it1)`, and `final refinement (itw)`.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK2-stages.png">
</figure>

In HADDOCK3, users have the freedom to configure docking workflows into
functional pipelines by combining the different HADDOCK3 modules, thus
adapting the workflows to their projects. HADDOCK3 has therefore developed to
truthfully work like a puzzle of many pieces (simulation modules) that users can
combine freely. To this end, the old HADDOCK machinery has been modularized,
and several new modules added, including third-party software additions. As a
result, the modularization achieved in HADDOCK3 allows users to duplicate steps
within one workflow (e.g., to repeat twice the `it1` stage of the HADDOCK2.x
rigid workflow).
Note that, for simplification purposes, at this time, not all functionalities of
HADDOCK2.x have been ported to HADDOCK3, which does not (yet) support NMR RDC,
PCS and diffusion anisotropy restraints, cryo-EM restraints and coarse-graining.
Any type of information that can be converted into ambiguous interaction
restraints can, however, be used in HADDOCK3, which also supports the
*ab initio* docking modes of HADDOCK.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK3-workflow-scheme.png">
</figure>

To keep HADDOCK3 modules organized, we catalogued them into several
categories. However, there are no constraints on piping modules of different
categories.

The main module categories are "topology", "sampling", "refinement",
"scoring", and "analysis". There is no limit to how many modules can belong to a
category. Modules are added as developed, and new categories will be created
if/when needed. You can access the HADDOCK3 documentation page for the list of
all categories and modules. Below is a summary of the available modules:

* **Topology modules**
    * `topoaa`: *generates the all-atom topologies for the CNS engine.*
* **Sampling modules**
    * `rigidbody`: *Rigid body energy minimization with CNS (`it0` in haddock2.x).*
    * `lightdock`: *Third-party glow-worm swam optimization docking software.*
* **Model refinement modules**
    * `flexref`: *Semi-flexible refinement using a simulated annealing protocol through molecular dynamics simulations in torsion angle space (`it1` in haddock2.x).*
    * `emref`: *Refinement by energy minimisation (`itw` EM only in haddock2.4).*
    * `mdref`: *Refinement by a short molecular dynamics simulation in explicit solvent (`itw` in haddock2.X).*
* **Scoring modules**
    * `emscoring`: *scoring of a complex performing a short EM (builds the topology and all missing atoms).*
    * `mdscoring`: *scoring of a complex performing a short MD in explicit solvent + EM (builds the topology and all missing atoms).*
* **Analysis modules**
    * `alascan`: *Performs a systematic (or user-define) alanine scanning mutagenesis of interface residues.*
    * `caprieval`: *Calculates CAPRI metrics (i-RMSD, l-RMSD, Fnat, DockQ) with respect to the top-scoring model or reference structure if provided.*
    * `clustfcc`: *Clusters models based on the fraction of common contacts (FCC)*
    * `clustrmsd`: *Clusters models based on pairwise RMSD matrix calculated with the `rmsdmatrix` module.*
    * `contactmap`: *Generate contact matrices of both intra- and intermolecular contacts and a chordchart of intermolecular contacts.*
    * `rmsdmatrix`: *Calculates the pairwise RMSD matrix between all the models generated in the previous step.*
    * `ilrmsdmatrix`: *Calculates the pairwise interface-ligand-RMSD (il-RMSD) matrix between all the models generated in the previous step.*
    * `seletop`: *Selects the top N models from the previous step.*
    * `seletopclusts`: *Selects the top N clusters from the previous step.*

The HADDOCK3 workflows are defined in simple configuration text files, similar to the TOML format but with extra features.
Contrary to HADDOCK2.X which follows a rigid (yet highly parameterisable)
procedure, in HADDOCK3, you can create your own simulation workflows by
combining a multitude of independent modules that perform specialized tasks.



---
# **Software and data setup**

In order to follow this tutorial we will start Jupyter session in Colab or other resources, install haddock3 and download the data for the tutorial.

To run this tutorial on Colab we recommend changing the runtime type to use one of the TPU option, which will give you access to more cores. The haddock3 docking run described in this tutorial should complete in about 15 minutes.

For this follow the following steps (comment out the `pip install haddock3` line if you already have a working installation):

In [1]:
#@title 1. Install haddock3, BioPython and py3Dmol
#@markdown Execute this cell to download the required packages.
!pip install haddock3==2026.7.0 --quiet
!pip install py3Dmol --quiet
!pip install BioPython --quiet

In [15]:
#@title 1. Download tutorial data and setup environment
#@markdown Execute this cell to download the tutorial data.
!wget https://surfdrive.surf.nl/public.php/dav/files/3GE8k07b8EtuVK8 -O HADDOCK3-protein-peptide.zip
!unzip HADDOCK3-protein-peptide.zip
!\rm HADDOCK3-protein-peptide.zip
import os
from pathlib import Path
BASE_DIR = Path.cwd()
PROJECT_NAME = 'HADDOCK3-protein-peptide-notebook'
PROJECT_DIR = BASE_DIR / PROJECT_NAME
os.chdir(PROJECT_DIR)

--2026-08-17 14:05:47--  https://surfdrive.surf.nl/public.php/dav/files/3GE8k07b8EtuVK8
surfdrive.surf.nl (surfdrive.surf.nl) çözümleniyor... 2001:610:10a:2:0:a11:da7a:5afe, 2001:610:10b:2:0:a11:da7a:5afe, 145.107.56.140, ...
surfdrive.surf.nl (surfdrive.surf.nl)[2001:610:10a:2:0:a11:da7a:5afe]:443 bağlanılıyor... bağlantı kuruldu.
HTTP isteği gönderildi, yanıt bekleniyor... 200 OK
Uzunluk: 241474807 (230M) [application/zip]
Kayıt yeri: `HADDOCK3-protein-peptide.zip'

HADDOCK3-protein-pe 100%[===================>] 230,29M   502KB/s    içinde 6m 44s

2026-08-17 14:12:31 (584 KB/s) - `HADDOCK3-protein-peptide.zip' kaydedildi [241474807/241474807]

Archive:  HADDOCK3-protein-peptide.zip
   creating: HADDOCK3-protein-peptide/
   creating: HADDOCK3-protein-peptide/workflows/
  inflating: HADDOCK3-protein-peptide/workflows/protein_peptide_docking_best_practices.cfg  
  inflating: HADDOCK3-protein-peptide/workflows/protein_peptide_docking.cfg  
   creating: HADDOCK3-protein-peptide/restraints

FileNotFoundError: [Errno 2] No such file or directory: '/Users/eceokur/Downloads/HADDOCK3-protein-peptide/HADDOCK3-protein-peptide-notebook'

ecompressing the file will create the `HADDOCK3-protein-peptide` directory with the following subdirectories and items:

* `pdbs`: Contains the pre-processed PDB files, both the docking input, and bound reference.
* `restraints`: Contains the interface information and the correspond restraint files for HADDOCK.
* `runs`: : Contains pre-computed docking results for each scenario defined in `workflows` directory, useful for comparison with your own run, or if you prefer to skip the computationally intensive steps.
* `scripts`: Contains two analysis scripts used in this tutorial.
* `workflows`: Contains HADDOCK3 workflow used for the docking.
----


---
# Preparing PDB files for docking

The accuracy of docking results in HADDOCK3 depends heavily on the quality of the input structures. In this section, we will prepare the PDB files of the protein and peptide for docking. The protein model is created using AlphaFold, and multiple conformations of the peptide are generated using [PyMOL](https://www.pymol.org/). We will use `pdb-tools`to manipulate the structures to make them HADDOCK-ready, e.g. to renumber residues, assign chain IDs and generate ensemble file. By default, `pdb-tools` are being installed on your machine together with haddock3. `pdb-tools` documentation is available [here](https://www.bonvinlab.org/pdb-tools/).

Note: that pdb-tools is also available as a [web service](https://wenmr.science.uu.nl/pdbtools/).

Note: Before starting to work on the tutorial, make sure to activate an appropriate virtual environment. If haddock3 was installed using `conda`run the following cell:



In [16]:
! conda activate haddock3


CondaError: Run 'conda init' before 'conda activate'



For more information about accepted file formats and preparation steps, refer to the [HADDOCK3 user manual – structure requirements](https://www.bonvinlab.org/haddock3-user-manual/structure_requirements.html).

# Preparing the protein structure

One of the easiest ways to find information about a protein, including its structure, is to explore the UniProt database. [UniProt](https://www.uniprot.org/) provides detailed protein sequence and functional data, including annotations, sequence features, and links to structural information.

__*Open the UniProt home page (https://www.uniprot.org) and search for “mouse MDM2”.*__

You should see the entry “P23804 · MDM2_MOUSE”. Feel free to take you time and explore the page.

__*Click on the section ‘Structure’ (left side of the page) or scroll down until you reach it.*__

This protein has no experimentally solved 3D structure, only AlphaFold model is available. This model model covers the full-length sequence of MDM2, but for docking we only need its p53-binding domain. This domain corresponds to residues 26 to 109. Check out [Family & Domains section](https://www.uniprot.org/uniprotkb/P23804/entry#family_and_domains) of the UniProt to see all other regions of the protein. The remaining regions, particularly the disordered one, are known not to interact with the peptide, so it’s a good idea remove them, both to make the docking problem easier, and to reduce the computational cost of the docking.

__*Click the download icon at the right end of the yellow ribbon to obtain this model. At the time this tutorial was created, the file name was AF-P23804-F1-model_v6.pdb. With future updates to the AlphaFold Database, the version tag “v6” will change.*__

__*Move downloaded model AF-P23804-F1-model_v4.pdb to your work directory, e.g. HADDOCK3-protein-peptide/*__

To prepare this model for docking, we will:

1 Keep only the coordinates lines, i.e and remove REMARK and other irrelevant lines (`pdb_keepcoord`),
2. Keep only residues of interest (`pdb_selres`),
3. Assign chain ID (`pdb_chain`), and
4. Apply pdb-formatting to the file (`pdb_tidy`).



In [17]:
! pdb_keepcoord AF-P23804-F1-model_v6.pdb | pdb_selres -26:109 | pdb_chain -A | pdb_tidy -strict > AF_MDM2_26_109.pdb



*Note* when working with the experimentally solved structures, preparation algorithm would be different, e.g. we would use `pdb_delhetatm` and `pdb_selaltloc`.

*Note* `pdb_tidy` attempts to correct formatting only, e.g. ensure each line is long enough (padded with spaces), not the actual content of the PDB file.

It’s a good practice to verify resulting structure visually using PyMOL. Start PyMOL and load the PDB file as followed:

__*File menu -> Open -> select HADDOCK3-protein-peptide/AF_MDM2_26_109.pdb*__

Feel free to compare this structure with initial AlphaFold model:
__*File menu -> Open -> select HADDOCK3-protein-peptide/AF-P23804-F1-model_v4.pdb*__

In [18]:
! ls

AF-P23804-F1-model_v6.pdb pdbs                      scripts
AF_MDM2_26_109.pdb        restraints                workflows
HADDOCK3-protein-peptide  runs


In [20]:
from haddock.libs.libnotebooks import align_full
#@title View the top ranking model from the best cluster
#@markdown It will be shown superimposed onto the reference structure fitted only on the second receptor and the DNA
# Define paths to PDB files to be aligned
# Here we are overlaying the reference and 1st model from the 1st cluster
path1 = PROJECT_DIR / "AF_MDM2_26_109.pdb"
path2 = PROJECT_DIR / "AF-P23804-F1-model_v6.pdb"

# Align molecules by chain and display the result.
# For the full alignement put all chain IDs in the `chains` variable.
align_full(str(path1), str(path2), chains = ['B','C'])

Error: File not found at /Users/eceokur/Downloads/HADDOCK3-protein-peptide/HADDOCK3-protein-peptide-notebook/AF_MDM2_26_109.pdb
Error: File not found at /Users/eceokur/Downloads/HADDOCK3-protein-peptide/HADDOCK3-protein-peptide-notebook/AF-P23804-F1-model_v6.pdb
Failed to load one or both PDB files


(<py3Dmol.view at 0x110f28790>, None, {})